# CNN-BiLSTM Training (Mumbi)

CNN (TDSConvCTC) encoder with BiLSTM decoder for EMG-based keystroke decoding.

| Condition | Val CER | Test CER |
|---|---|---|
| Spectrogram (baseline) | 15.8% | 19.0% |
| Mel spectrogram / biophys (8ch, 1kHz) | 17.9% | 21.4% |
| AE-reconstructed EMG (recons v3) | 62.2% | 69.2% |

> **GPU used:** NVIDIA L4 (Google Colab)

## 1. Setup

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# ── Detect environment ─────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

env = 'Google Colab' if IN_COLAB else 'local Jupyter'
print(f'Environment: {env}')

# ── Repo root setup ────────────────────────────────────────────────────────
if IN_COLAB:
    REPO_ROOT = Path('/content/C247_mumbi_kai_jonathan_ben')
    if not REPO_ROOT.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/Mumbzzz/C247_mumbi_kai_jonathan_ben',
             str(REPO_ROOT)],
            check=True
        )
else:
    # Walk up from cwd to find repo root (identified by setup.py)
    p = Path.cwd()
    while p != p.parent:
        if (p / 'setup.py').exists():
            REPO_ROOT = p
            break
        p = p.parent
    else:
        REPO_ROOT = Path.cwd()
        print('Warning: could not locate repo root, using cwd')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root: {REPO_ROOT}')

In [ ]:
import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q protobuf==3.20.3 kenlm unidecode python-Levenshtein torchmetrics h5py==3.11.0
!pip install -q -e .

## 2. Configure Output Paths

In Colab, results and checkpoints are symlinked to Google Drive for persistence across sessions. Locally, they use repo-relative `results/` and `Playground_Mumbi/checkpoints/` directories.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    RESULTS_DIR     = Path('/content/drive/MyDrive/Classes/ECE247A/C247_results')
    CHECKPOINTS_DIR = Path('/content/drive/MyDrive/Classes/ECE247A/C247_checkpoints')
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

    for src, dst in [
        (str(RESULTS_DIR),     'results'),
        (str(CHECKPOINTS_DIR), 'Playground_Mumbi/checkpoints'),
    ]:
        dst_path = REPO_ROOT / dst
        if dst_path.is_symlink():  dst_path.unlink()
        elif dst_path.exists():    shutil.rmtree(dst_path)
        dst_path.symlink_to(src)

    r = (REPO_ROOT / 'results').resolve()
    c = (REPO_ROOT / 'Playground_Mumbi' / 'checkpoints').resolve()
    print(f'Results  → {r}')
    print(f'Checkpts → {c}')
else:
    RESULTS_DIR     = REPO_ROOT / 'results'
    CHECKPOINTS_DIR = REPO_ROOT / 'Playground_Mumbi' / 'checkpoints'
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Results  : {RESULTS_DIR}')
    print(f'Checkpts : {CHECKPOINTS_DIR}')

## 3. Download Data

Raw EMG (~4.4 GB) and AE-reconstructed EMG (~125 MB) are downloaded automatically in Colab. For local runs, download manually using the links printed and place files in `data/89335547/` and `data/89335547_recons_v3/`.

In [ ]:
DATA_DIR = REPO_ROOT / 'data' / '89335547'

if DATA_DIR.exists():
    print(f'Raw EMG data already present at {DATA_DIR}')
elif IN_COLAB:
    !wget -O 89335547.zip https://ucla.box.com/shared/static/dy28c8fne2d2bypyc0dz6xnfmmohgoyq
    !unzip 89335547.zip
    !mkdir -p data/89335547
    !mv 89335547/* data/89335547/
    !rm 89335547.zip
else:
    print('Raw EMG data not found.')
    print('Download: https://ucla.box.com/shared/static/dy28c8fne2d2bypyc0dz6xnfmmohgoyq')
    print(f'Extract and place files in: {DATA_DIR}')

In [ ]:
RECONS_DIR = REPO_ROOT / 'data' / '89335547_recons_v3'

if RECONS_DIR.exists():
    print(f'Recons data already present at {RECONS_DIR}')
elif IN_COLAB:
    !wget -O 89335547_recons_v3.zip https://ucla.box.com/shared/static/rtik65u7bhrllelm99po9ebhowuhnmaf
    !unzip 89335547_recons_v3.zip
    !mkdir -p data/89335547_recons_v3
    !mv 89335547_recons_v3/* data/89335547_recons_v3/
    !rm 89335547_recons_v3.zip
else:
    print('AE-reconstructed EMG data not found.')
    print('Download: https://ucla.box.com/shared/static/rtik65u7bhrllelm99po9ebhowuhnmaf')
    print(f'Extract and place files in: {RECONS_DIR}')

## 4. Hyperparameter Tuning

Bayesian search over CNN-BiLSTM hyperparameters: 8 trial sessions, 20 coarse trials + 10 fine trials. Best hyperparameters saved to `Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm.yaml`.

In [ ]:
!python -m Playground_Mumbi.hyperparam_tuner \
      --trial-sessions 8 \
      --coarse-trials 20 \
      --coarse-epochs 60 \
      --fine-trials 10 \
      --fine-epochs 80 \
      --trial-timeout 300 \
      --early-stopping-patience 10 \
      --batch-size 32

if IN_COLAB:
    from google.colab import files
    files.download('Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm.yaml')

## 5. CNN-BiLSTM Training

### 5a. Spectrogram (Baseline)

In [ ]:
# Best val CER: 15.8%  |  Best test CER: 19.0%
!python -m Playground_Mumbi.train \
    --model cnn_lstm \
    --epochs 150 \
    --from-hyperparams Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm.yaml \
    --batch-size 32 \
    --num-workers 2

if IN_COLAB:
    from google.colab import files
    files.download('results/results_summary_CNN_LSTM.csv')
    files.download('results/results_curves_CNN_LSTM.csv')
    files.download('Playground_Mumbi/checkpoints/final_models/best_cnnlstm.pt')

### 5b. Biophysical Features (Mel, 8ch, 1kHz)

In [ ]:
# Best val CER: 17.9%  |  Best test CER: 21.4%
!python -m Playground_Mumbi.train \
    --model cnn_lstm \
    --biophys \
    --from-hyperparams Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm.yaml \
    --epochs 150 \
    --batch-size 32 \
    --num-workers 2

if IN_COLAB:
    from google.colab import files
    files.download('Playground_Mumbi/checkpoints/best_cnnlstm_biophys.pt')
    files.download('results/results_summary_CNN_LSTM.csv')
    files.download('results/results_curves_CNN_LSTM.csv')

### 5c. AE-Reconstructed EMG (recons v3)

In [ ]:
# Optional: re-tune hyperparameters specifically for recons data
# !python -m Playground_Mumbi.hyperparam_tuner_recons \
#       --trial-sessions 8 \
#       --coarse-trials 20 \
#       --coarse-epochs 60 \
#       --fine-trials 10 \
#       --fine-epochs 80 \
#       --trial-timeout 300 \
#       --early-stopping-patience 10 \
#       --batch-size 32
# if IN_COLAB:
#     from google.colab import files
#     files.download('Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm_recons.yaml')

In [ ]:
# Best val CER: 62.2%  |  Best test CER: 69.2%
!python -m Playground_Mumbi.train_recons \
    --from-hyperparams Playground_Mumbi/checkpoints/best_hyperparams_cnn_lstm.yaml \
    --epochs 150 \
    --batch-size 32 \
    --num-workers 2

if IN_COLAB:
    from google.colab import files
    files.download('Playground_Mumbi/checkpoints/best_cnnlstm_recons_v3.pt')
    files.download('results/results_summary_CNN_LSTM.csv')
    files.download('results/results_curves_CNN_LSTM.csv')